# Setup


In [ ]:
!pip install accelerate -U
!pip install crcmod
!gsutil cp  gs://hien7613storage2/kgt5_wn18rr_data.pt /content/


Copying gs://hien7613storage2/kgt5_wn18rr_data.pt...
/
Operation completed over 1 objects/130.3 MiB.                                    


# Dataset

In [ ]:
%%capture

import torch
from transformers import T5Tokenizer
import numpy as np

import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

## Hop1Index

In [ ]:

class Hop1Index:
    def __init__(self, triples, num_entities, key_col=0, max_context_size=64):
        self.max_context_size = max_context_size
        self.shuffle = False
        self.key_col = key_col
        self.triples = triples[triples[:, key_col].argsort()]
        keys, values_offset = np.unique(
            self.triples[:, key_col], axis=0, return_index=True
        )
        values_offset = np.append(values_offset, len(self.triples))
        self.keys = keys
        self.values_offset = values_offset

        self.key_to_start = -1 * np.ones(num_entities, dtype=int)
        self.key_to_start[keys] = values_offset[:-1]
        self.key_to_end = -1 * np.ones(num_entities, dtype=int)
        self.key_to_end[keys] = values_offset[1:]

    def __getitem__(self, item, rel_id=None):
        start = self.key_to_start[item]
        end = self.key_to_end[item]
        context = self.triples[start:end, [1, 2 - self.key_col]]
        if rel_id is not None:
            context = context[context[:,0] == rel_id][:,1]
        if len(context) > self.max_context_size:
            ids = np.random.choice(len(context), self.max_context_size, replace=False)
            context = context[ids]
        if self.shuffle:
            np.random.shuffle(context)
        return context

    def get_context(self, item, rel_id=None):
        return self.__getitem__(item, rel_id)



## KGCDataset

In [ ]:
kgt5_data = torch.load('/content/kgt5_wn18rr_data.pt')

In [ ]:

tokenizer = T5Tokenizer.from_pretrained('t5-small', padding=True)

def _tokenize( x):
    return tokenizer(x, return_tensors="pt")['input_ids'][0][:-1]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
from numpy import pi
import warnings
from torch import nn
import torch
import torch.nn.functional as F
import warnings

class RotatE:
    def __init__(self, k, max_rel_size=None, entity_embedding=None, relation_embedding=None):
        self.internal_k = 2 * k
        self.max_rel_size = max_rel_size
        self.entity_embedding = entity_embedding
        self.relation_embedding = relation_embedding

    def __call__(self, e_s_id, e_p_id):
        e_s = self.entity_embedding[e_s_id]
        e_p = self.relation_embedding[e_p_id]
        e_s_real, e_s_img = torch.chunk(e_s, 2, axis=0)
        theta_pred = e_p

        embedding_range = (6 / (self.internal_k * self.max_rel_size)) ** 0.5
        e_p_real = torch.cos(theta_pred / (embedding_range / pi))
        e_p_img = torch.sin(theta_pred / (embedding_range / pi))

        e_o_real = e_s_real * e_p_real - e_s_img * e_p_img
        e_o_img = e_s_real * e_p_img + e_s_img * e_p_real
        return torch.cat([e_o_real, e_o_img], axis=0)

structal_model = RotatE(k=350, entity_embedding=kgt5_data['struct_ent_emb'], relation_embedding=kgt5_data['struct_rel_emb'], max_rel_size=237)


In [ ]:
import numpy as np
from torch.utils.data import Dataset
from typing import Dict, Optional, Union, Tuple, List
import random


class KGCDataset(Dataset):
    def __init__(self, num_ents=14541, structal_model=None):
        self.num_ents = num_ents
        self.structal_model = structal_model
        index = 0
        # Fb15k wn18rr
        self.id_triplets ={
            'train': kgt5_data['train_triplet_id'],
            'valid': kgt5_data['valid_triplet_id'],
            'test': kgt5_data['test_triplet_id']
        }
        self.tokens_triplets ={
            'train': kgt5_data['train_triplet_tokens'],
            'valid': kgt5_data['valid_triplet_tokens'],
            'test': kgt5_data['test_triplet_tokens']
        }

        self.get_neigs_0 ={
            'train': Hop1Index(
                kgt5_data['train_triplet_id'],
                self.num_ents, 0),
            'valid': Hop1Index(
                kgt5_data['valid_triplet_id'],
                self.num_ents, 0),
            'test': Hop1Index(
                kgt5_data['test_triplet_id'],
                self.num_ents, 0)
        }
        self.get_neigs_2 ={
            'train': Hop1Index(
                kgt5_data['train_triplet_id'],
                self.num_ents, 2),
            'valid': Hop1Index(
                kgt5_data['valid_triplet_id'],
                self.num_ents, 2),
            'test': Hop1Index(
                kgt5_data['test_triplet_id'],
                self.num_ents, 2)
        }

        self.sep = _tokenize(' | ')
        self.mask_token = _tokenize('<extra_id_90>')
        self.eos_token = torch.tensor([tokenizer.eos_token_id])

        self.predict_head_token = _tokenize('predict head :')
        self.predict_tail_token = _tokenize('predict tail :')
        self.start_decs_token = _tokenize('[')
        self.end_decs_token = _tokenize(']')
        self.inversion_token = _tokenize('inversion of ')
        self.empty_token = torch.tensor([], dtype=torch.int)
        self.set_ent_id = set(range(self.num_ents))
        self.p_dropout = 0.#1#05

    def __getitem__(self, idx):
        return self.get(idx, split=self.split)
    def __len__(self, split='train'):
        return len(self.tokens_triplets[split])

    def get(self, idx: int, split: str = "train", full_mask_part_idx=None):
        head_lbl, relation, tail_lbl = self.tokens_triplets[split][idx]
        head_id, rel_id, tail_id = self.id_triplets[split][idx]

        if full_mask_part_idx is None:
          full_mask_part_idx = 2 if random.randint(0, 1) else 0

        inversion = False

        if full_mask_part_idx:
          source = [
              self.predict_tail_token if not inversion else self.predict_head_token,
              head_lbl,
              self.sep,
              self.inversion_token if inversion else self.empty_token,
              relation,
          ]
          target = [tail_lbl]
          label_id = tail_id
          neighboors_0 = self.get_neigs_0[split][head_id]
          neighboors_0 = neighboors_0[(neighboors_0[:,0]!=rel_id) | (neighboors_0[:,1]!=tail_id)]
          neighboors_2 = self.get_neigs_2[split][head_id]
          neighboors_2 = neighboors_2[(neighboors_2[:,0]!=rel_id) | (neighboors_2[:,1]!=tail_id)]
        else:
          source = [
              self.predict_head_token if not inversion else self.predict_tail_token,
              tail_lbl,
              self.sep,
              self.inversion_token if inversion else self.empty_token,
              relation,
          ]
          target = [head_lbl]
          label_id = head_id
          neighboors_0 = self.get_neigs_0[split][tail_id]
          neighboors_0 = neighboors_0[(neighboors_0[:,0]!=rel_id) | (neighboors_0[:,1]!=head_id)]
          neighboors_2 = self.get_neigs_2[split][tail_id]
          neighboors_2 = neighboors_2[(neighboors_2[:,0]!=rel_id) | (neighboors_2[:,1]!=head_id)]

        target_ent_embeddings = []
        neighboors_embeddings = []
        for rel_n_id, ent_n_id in neighboors_0:

          ent_n_embedding = self.structal_model.entity_embedding[ent_n_id]
          rel_n_embedding = self.structal_model.relation_embedding[rel_n_id]
          target_ent_embedding = self.structal_model(ent_n_id, rel_n_id)
          neighboors_embeddings.append(torch.cat([ent_n_embedding, rel_n_embedding]))
          target_ent_embeddings.append(target_ent_embedding)
        for rel_n_id, ent_n_id in neighboors_2:

          ent_n_embedding = self.structal_model.entity_embedding[ent_n_id]
          rel_n_embedding = self.structal_model.relation_embedding[rel_n_id]
          target_ent_embedding = self.structal_model(ent_n_id, rel_n_id)
          neighboors_embeddings.append(torch.cat([ent_n_embedding, -rel_n_embedding]))
          target_ent_embeddings.append(target_ent_embedding)

        if len(neighboors_embeddings):
          neighboors_embeddings = torch.stack(neighboors_embeddings)
          target_ent_embeddings = torch.stack(target_ent_embeddings)
          neighboors_embeddings_mask = torch.ones(len(neighboors_embeddings))
        else:
          neighboors_embeddings_mask = torch.zeros([1])
          neighboors_embeddings = torch.zeros([1, 256+128])
          target_ent_embeddings = torch.zeros([1, 256])


        source.append(self.eos_token)
        target.append(self.eos_token)
        source = torch.cat(source)
        target = torch.cat(target)

        attention_mask = torch.ones_like(source)
        rand = torch.rand_like(attention_mask.float())
        dropout = torch.logical_not(rand < self.p_dropout).long()
        dropout[(source == self.start_decs_token[0]) | (source == self.end_decs_token[0])] = 1
        dropout[:4]=1
        inversion_len = len(self.inversion_token if inversion else self.empty_token)
        relation_len = len(relation)
        dropout[-relation_len-inversion_len:-relation_len]=1
        attention_mask = attention_mask * dropout


        output = {
            "input_ids": source,
            "attention_mask": attention_mask,
            "labels": target,
            'neighboors_embeddings': neighboors_embeddings,
            'neighboors_embeddings_mask': neighboors_embeddings_mask,
            'target_ent_embeddings': target_ent_embeddings,
            'triplet': self.id_triplets[split][idx],
        }
        return output



class SplitDatasetWrapper:
    def __init__(self, dataset, split, full_mask_part_idx=None):
        self.dataset = dataset
        self.split = split
        self.full_mask_part_idx = full_mask_part_idx
    def __getitem__(self, idx):
        return self.dataset.get(idx, self.split, self.full_mask_part_idx)
    def __len__(self):
        return self.dataset.__len__(split=self.split)

dataset = KGCDataset(num_ents=40943, structal_model=structal_model)
train_dataset = SplitDatasetWrapper(dataset, split="train")
valid_dataset = SplitDatasetWrapper(dataset, split="valid")
test_dataset = SplitDatasetWrapper(dataset, split="test")


head_test_dataset = SplitDatasetWrapper(dataset, split="test", full_mask_part_idx=0)
tail_test_dataset = SplitDatasetWrapper(dataset, split="test", full_mask_part_idx=2)

head_valid_dataset = SplitDatasetWrapper(dataset, split="valid", full_mask_part_idx=0)
tail_valid_dataset = SplitDatasetWrapper(dataset, split="valid", full_mask_part_idx=2)

head_train_dataset = SplitDatasetWrapper(dataset, split="train", full_mask_part_idx=0)
tail_train_dataset = SplitDatasetWrapper(dataset, split="train", full_mask_part_idx=2)


ext_get_neigs_0 ={
    'train': Hop1Index(
        kgt5_data['train_triplet_id'],
        dataset.num_ents, 0, max_context_size=1e10),
    'valid': Hop1Index(
        kgt5_data['valid_triplet_id'],
        dataset.num_ents, 0, max_context_size=1e10),
    'test': Hop1Index(
        kgt5_data['test_triplet_id'],
        dataset.num_ents, 0, max_context_size=1e10),
}

ext_get_neigs_2 ={
    'train': Hop1Index(
        kgt5_data['train_triplet_id'],
        dataset.num_ents, 2, max_context_size=1e10),
    'valid': Hop1Index(
        kgt5_data['valid_triplet_id'],
        dataset.num_ents, 2, max_context_size=1e10),
    'test': Hop1Index(
        kgt5_data['test_triplet_id'],
        dataset.num_ents, 2, max_context_size=1e10),
}

# get all ground truth
def get_neigs2(ent_id, rel_id):
  n_train = ext_get_neigs_2['train'].__getitem__(ent_id, rel_id)
  n_valid = ext_get_neigs_2['valid'].__getitem__(ent_id, rel_id)
  n_test = ext_get_neigs_2['test'].__getitem__(ent_id, rel_id)
  return [n_train, n_test]
def get_neigs0(ent_id, rel_id):
  n_train = ext_get_neigs_0['train'].__getitem__(ent_id, rel_id)
  n_valid = ext_get_neigs_0['valid'].__getitem__(ent_id, rel_id)
  n_test = ext_get_neigs_0['test'].__getitem__(ent_id, rel_id)
  return [n_train, n_test]



# model

In [ ]:
model_name='t5-small'
model = EditedT5.from_pretrained(model_name)
model.to('cuda')
print('')

In [ ]:
!gsutil cp  gs://hien7613storage2/kgt5_wn18rr_finalx16.pt /content/
state_dict = torch.load('kgt5_wn18rr_finalx16.pt', map_location=torch.device('cpu'))
model.load_state_dict(state_dict)


<All keys matched successfully>

## DataCollatorForSeq2Seq

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class DataCollatorForSeq2Seq:
    model= None
    padding= True
    max_length= None
    pad_to_multiple_of=None
    label_pad_token_id= -100
    data_names = None
    def __init__(self, tokenizer, model=None, padding=True, max_length=None, pad_to_multiple_of=None, label_pad_token_id=-100,data_names=None):
        self.tokenizer = tokenizer
        self.model = model
        self.data_names = data_names
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features):
        features2 = {}
        for name in self.data_names:
          if name == 'triplet':
            continue
          if name in ['labels','filter_id']:
            padding_value=self.label_pad_token_id
          else:
            padding_value=self.tokenizer.pad_token_id
          x_features = [feature[name] for feature in features]
          # try:
          features2[name] = torch.nn.utils.rnn.pad_sequence(x_features, batch_first=True, padding_value=padding_value)
          # except:
          #   for m in x_features:
          #     print(m.shape)
            # print(x_features)
        if self.model is not None and hasattr(self.model, "prepare_decoder_input_ids_from_labels"):
            decoder_input_ids = self.model.prepare_decoder_input_ids_from_labels(labels=features2["labels"])
            features2["decoder_input_ids"] = decoder_input_ids
        return features2


data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, data_names=list(train_dataset[0].keys()))


In [ ]:
# train_dataset[0]

In [ ]:
# tokenizer('DataCollatorForSeq2Seq(tokenizer, model=model, data_names=list(train_dataset[0].keys()))')

In [ ]:
# data = data_collator([test_dataset[8], test_dataset[8]])
# for k, v in data.items():
#   data[k] = v.to('cuda')
#   # print(k, v)
#   pass

In [ ]:
# tokenizer.decode(data['decoder_input_ids'][0])

In [ ]:
# optimizer = AdamW(model.parameters(), lr=1e-4)

# output = model(**data)
# print(output.loss)
# loss = output.loss
# loss.backward()
# optimizer.step()

In [ ]:
# print(output.logits)

In [ ]:
# for k, v in data.items():
#   print(k, v.shape)

#train

In [ ]:
from transformers import Seq2SeqTrainingArguments, TrainingArguments
from transformers import Seq2SeqTrainer
batch_size= 32*4

args = Seq2SeqTrainingArguments(
    "kgt5-rotatE",
    dataloader_num_workers=8,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,

    num_train_epochs=1000,
    do_eval=True,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy='epoch',

    learning_rate=2e-5,
    torch_compile=True,
    tf32=True,
    # push_to_hub=True,
    # hub_strategy='checkpoint',

)

In [ ]:
from transformers import Seq2SeqTrainer
from transformers import EarlyStoppingCallback
from transformers import AdamW

# optimizer = AdamW(model.parameters(), lr=1e-4)
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    # optimizers=(optimizer, None),
)

In [ ]:
trainer.train()
# baaed1dc0ef02b02dff291c8e0cfacf571bff2f9

/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Epoch,Training Loss,Validation Loss
1,0.191800,0.390279
2,0.216500,0.857914
3,0.136700,0.644782
4,0.206900,0.465613
5,0.181300,0.487717
6,0.157100,0.455841
7,0.154300,1.142595
8,0.150800,0.648863
9,0.181500,0.620714
10,0.166000,0.666668


/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os

KeyboardInterrupt: 

In [ ]:
import torch

trainer.model.eval()
model_state_dict = trainer.model.state_dict()
torch.save(model_state_dict, '/content/kgt5_wn18rr_finalx16.pt')

!gsutil -o GSUtil:parallel_composite_upload_threshold=150M cp  /content/kgt5_wn18rr_finalx16.pt gs://hien7613storage2/

# New Eval

## setup eval

In [ ]:
# ent2text
from tqdm.auto import tqdm
import os

if not os.path.exists('/content/entityid2description.txt'):
  !wget https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/entityid2description.txt
path = "/content/entityid2description.txt"

ent2decs = {}
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    ent, text = line.strip().split('\t')
    ent2decs[ent] = _tokenize(text)[:64]

if not os.path.exists('/content/entityid2name.txt'):
  !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/entityid2name.txt
path = "/content/entityid2name.txt"

ent2text = {}
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    ent, text = line.strip().split('\t')
    ename, etype, _ = text.split(',')
    text = etype + ', ' + ename + ', '
    ent2text[ent] = torch.cat([_tokenize(text), ent2decs[ent]])



# ent2id
from tqdm.auto import tqdm
import os
if not os.path.exists('/content/entity2id.txt'):
  !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/entity2id.txt
path = "/content/entity2id.txt"

ent2id = {}
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    # print(line.strip().split('\t'))
    ent, id = line.strip().split('\t')
    ent2id[ent] = int(id)


entid2text = [0]*len(ent2id)
for ent in tqdm(range(len(ent2id))):
  entid2text[ent] = [0] + ent2text[str(ent)].tolist() + [1]

# ent_name_decode_list = []
# for target in tqdm(entid2text):
#   ent_name_decode_list.append(tokenizer.decode(target[1:-1]))

Processing lines:   0%|          | 0/40944 [00:00<?, ?it/s]

Processing lines:   0%|          | 0/40944 [00:00<?, ?it/s]

Processing lines:   0%|          | 0/40944 [00:00<?, ?it/s]

  0%|          | 0/40943 [00:00<?, ?it/s]

In [ ]:
from typing import Dict, List
class Trie(object):
    def __init__(self, sequences: List[List[int]] = []):
        self.trie_dict = {}
        self.len = 0
        if sequences:
            for sequence in sequences:
                Trie._add_to_trie(sequence, self.trie_dict)
                self.len += 1
        self.append_trie = None
        self.bos_token_id = None
    def append(self, trie, bos_token_id):
        self.append_trie = trie
        self.bos_token_id = bos_token_id
    def add(self, sequence: List[int]):
        Trie._add_to_trie(sequence, self.trie_dict)
        self.len += 1
    def get(self, prefix_sequence: List[int]):
        return Trie._get_from_trie(prefix_sequence, self.trie_dict, self.append_trie, self.bos_token_id)
    @staticmethod
    def load_from_dict(trie_dict):
        trie = Trie()
        trie.trie_dict = trie_dict
        trie.len = sum(1 for _ in trie)
        return trie
    @staticmethod
    def _add_to_trie(sequence: List[int], trie_dict: Dict):
        if sequence:
            if sequence[0] not in trie_dict:
                trie_dict[sequence[0]] = {}
            Trie._add_to_trie(sequence[1:], trie_dict[sequence[0]])
    @staticmethod
    def _get_from_trie(
        prefix_sequence: List[int],
        trie_dict: Dict,
        append_trie=None,
        bos_token_id: int = None,
    ):
        if len(prefix_sequence) == 0:
            output = list(trie_dict.keys())
            if append_trie and bos_token_id in output:
                output.remove(bos_token_id)
                output += list(append_trie.trie_dict.keys())
            if len(output) == 0:
                return [0]
            return output
        elif prefix_sequence[0] in trie_dict:
            return Trie._get_from_trie(
                prefix_sequence[1:],
                trie_dict[prefix_sequence[0]],
                append_trie,
                bos_token_id,
            )
        else:
            if append_trie:
                return append_trie.get(prefix_sequence)
            else:
                return [0]
    def __iter__(self):
        def _traverse(prefix_sequence, trie_dict):
            if trie_dict:
                for next_token in trie_dict:
                    yield from _traverse(prefix_sequence + [next_token], trie_dict[next_token])
            else:
                yield prefix_sequence

        return _traverse([], self.trie_dict)
    def __len__(self):
        return self.len
    def __getitem__(self, value):
        return self.get(value)
trie = Trie(entid2text)

In [ ]:
import numpy as np
import pandas as pd
def _get_performance(ranks):
    ranks = np.array(ranks, dtype=np.float32)
    out = dict()
    out['mr'] = ranks.mean(axis=0)
    out['mrr'] = (1. / ranks).mean(axis=0)
    out['hit1'] = np.sum(ranks == 1, axis=0) / len(ranks)
    out['hit3'] = np.sum(ranks <= 3, axis=0) / len(ranks)
    out['hit10'] = np.sum(ranks <= 10, axis=0) / len(ranks)
    return out


def get_performance(model, tail_ranks, head_ranks):
    tail_out = _get_performance(tail_ranks)
    head_out = _get_performance(head_ranks)
    mr = np.array([tail_out['mr'], head_out['mr']])
    mrr = np.array([tail_out['mrr'], head_out['mrr']])
    hit1 = np.array([tail_out['hit1'], head_out['hit1']])
    hit3 = np.array([tail_out['hit3'], head_out['hit3']])
    hit10 = np.array([tail_out['hit10'], head_out['hit10']])
    val_mrr = mrr.mean().item()
    perf = {'mrr': mrr, 'mr': mr, 'hit@1': hit1, 'hit@3': hit3, 'hit@10': hit10}
    perf = pd.DataFrame(perf, index=['tail ranking', 'head ranking'])
    perf.loc['mean ranking'] = perf.mean(axis=0)
    for hit in ['hit@1', 'hit@3', 'hit@5', 'hit@10']:
        if hit in list(perf.columns):
            perf[hit] = perf[hit].apply(lambda x: '%.2f%%' % (x * 100))
    return perf



In [ ]:
import re
from collections import Counter
# from helper import get_performance
import numpy as np
import random

class RunEval:
    def __init__(self, configs, model, tokenizer, target_embeddings):
        self.configs = configs
        self.target_embeddings = target_embeddings
        self.configs = configs
        self.model = model
        self.tokenizer = tokenizer

    @torch.no_grad()
    def validation_step(self, batched_data, dataset_idx):
        input_ids = batched_data['input_ids'].to('cuda')
        attention_mask = batched_data['attention_mask'].to('cuda')
        labels = batched_data['labels']
        labels = torch.where(labels != -100, labels, tokenizer.pad_token_id)
        neighboors_embeddings=batched_data['neighboors_embeddings'].to('cuda')
        neighboors_embeddings_mask=batched_data['neighboors_embeddings_mask'].to('cuda')
        target_ent_embeddings=batched_data['target_ent_embeddings'].to('cuda')
        triple_id = batched_data['triplet'].numpy()

        self.get_neigs = get_neigs2 if dataset_idx == 0 else get_neigs0

        old_seqs = []
        ranks = torch.randint(self.configs.num_beams + 1, self.configs.n_ent, (len(labels),))
        for i in range(self.configs.num_beams):
          outputs = self.model.generate(
              input_ids=input_ids,
              attention_mask=attention_mask,
              return_dict_in_generate=True,
              # num_return_sequences=1,
              max_length=512,
              # num_beams=3,
              # eos_token_id=1,
              prefix_allowed_tokens_fn=lambda batch_idx, m_input_ids: self._next_candidate(batch_idx, m_input_ids, triple_id, dataset_idx, old_seqs),
              # bos_token_id=0,
              decoder_start_token_id=0,
              neighboors_embeddings=neighboors_embeddings,
              neighboors_embeddings_mask=neighboors_embeddings_mask,
              target_ent_embeddings=target_ent_embeddings,
          )
          pred = outputs.sequences.cpu()
          old_seqs.append(pred)
          pred = pred[:,1:]
          if pred.shape[1] > labels.shape[1]:
            pred = pred[:,:labels.shape[1]]
            cut_labels = labels
          else:
            cut_labels = labels[:,:pred.shape[1]]
          seq_match = (pred==cut_labels).all(1)
          new_ranks = torch.where(~seq_match, ranks, i+1)
          ranks = torch.min(ranks, new_ranks)
        # print(labels)
        # print(outputs.sequences.cpu()[1])
        # preds = tokenizer.batch_decode(outputs.sequences.cpu(), )
        # labels = tokenizer.batch_decode(batched_data['decoder_input_ids'], )
        # for pred, label in zip(preds, labels):
        #   print(pred)
        #   print(label)
        #   print('='*30)
        # raise 'sdfa'

        ranks = ranks.tolist()
        out = {'ranks': ranks}
        return out


    def _next_candidate(self, batch_idx, input_ids, triple_id, dataset_idx, old_seqs=None):
        input_ids = input_ids.cpu()
        # print(input_ids.tolist())
        # next = trie.get(input_ids.tolist())
        # print(next)
        # print('='*30)
        # return next
        if input_ids[-1] == 0 and len(input_ids) != 1:
            return [0]
        pred_ids = self.target_embeddings[triple_id[batch_idx][dataset_idx]]
        if len(input_ids) >= len(pred_ids):
            return [0]
        pred_id = int(pred_ids[len(input_ids)])
        all_gt_ids = torch.cat(self.get_neigs(triple_id[batch_idx][2-dataset_idx], triple_id[batch_idx][1]))

        all_gt_seq = torch.index_select(self.target_embeddings, 0, all_gt_ids)
        all_gt_seq_mask = (all_gt_seq[:, :len(input_ids)]==input_ids).all(1)
        all_gt_seq_tokens = all_gt_seq[:, len(input_ids)][all_gt_seq_mask]
        if len(old_seqs) > 0:
          old_seq = torch.nn.utils.rnn.pad_sequence([x[batch_idx] for x in old_seqs], batch_first=True, padding_value=0)
          if old_seq.shape[1] > len(input_ids):
            old_seq_mask = (old_seq[:, :len(input_ids)]==input_ids).all(1)
            old_seq_tokens = old_seq[:, len(input_ids)][old_seq_mask]
          else:
            old_seq_tokens = torch.tensor([], dtype=torch.int64)
        else:
          old_seq_tokens = torch.tensor([], dtype=torch.int64)
        all_gt_seq_tokens = set(torch.cat([all_gt_seq_tokens, old_seq_tokens]).tolist())
        pred_id = int(pred_ids[len(input_ids)])
        next_tokens = set(trie.get(input_ids.tolist())).difference(all_gt_seq_tokens)
        if pred_id in all_gt_seq_tokens:
          next_tokens.add(pred_id)
        if len(next_tokens) == 0:
          return [0]
        return list(next_tokens)

    def validation_epoch_end(self, outs):
        pred_tail_out, pred_head_out = outs
        agg_tail_out, agg_head_out, agg_total_out = dict(), dict(), dict()
        for out in pred_tail_out:
            for key, value in out.items():
                if key in agg_tail_out:
                    agg_tail_out[key] += value
                else:
                    agg_tail_out[key] = value
        for out in pred_head_out:
            for key, value in out.items():
                if key in agg_head_out:
                    agg_head_out[key] += value
                else:
                    agg_head_out[key] = value
        tail_ranks, head_ranks = agg_tail_out['ranks'], agg_head_out['ranks']
        del agg_tail_out['ranks']
        del agg_head_out['ranks']
        perf = get_performance(self, head_ranks, tail_ranks)
        print(perf)
        return perf





## run eval

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class DataCollatorForSeq2Seq:
    model= None
    padding= True
    max_length= None
    pad_to_multiple_of=None
    label_pad_token_id= -100
    data_names = None
    def __init__(self, tokenizer, model=None, padding=True, max_length=None, pad_to_multiple_of=None, label_pad_token_id=-100,data_names=None):
        self.tokenizer = tokenizer
        self.model = model
        self.data_names = data_names
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features):
        features2 = {}
        for name in self.data_names:
          # if name == 'triplet':
          #   continue
          if name in ['labels','filter_id']:
            padding_value=self.label_pad_token_id
          else:
            padding_value=self.tokenizer.pad_token_id
          x_features = [feature[name] for feature in features]
          features2[name] = torch.nn.utils.rnn.pad_sequence(x_features, batch_first=True, padding_value=padding_value)
        if self.model is not None and hasattr(self.model, "prepare_decoder_input_ids_from_labels"):
            decoder_input_ids = self.model.prepare_decoder_input_ids_from_labels(labels=features2["labels"])
            features2["decoder_input_ids"] = decoder_input_ids
        return features2


data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, data_names=list(train_dataset[0].keys()))


In [ ]:
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

tail_data_loader = DataLoader(tail_test_dataset,
                          batch_size=128,
                          shuffle=False,
                          collate_fn=data_collator,
                          # num_workers=8,
                          pin_memory=True)
head_data_loader = DataLoader(head_test_dataset,
                          batch_size=128,
                          shuffle=False,
                          # num_workers=8,
                          collate_fn=data_collator,
                          pin_memory=True)


class Configs:
    def __init__(self):
        self.num_beams = 1
        # self.num_beam_groups = 1
        self.num_return_sequences = 1
        self.max_length = 32
        self.n_ent = 40943
        self.n_rel = 11

configs = Configs()
runEval = RunEval(configs, model, tokenizer, entid2text_emb)

runEval.model.eval()
head_list_result = []
for data in tqdm(head_data_loader):
    # pass
    rank_rs = runEval.validation_step(data, 0)
    head_list_result.append(rank_rs)
    # break
tail_list_result = []
for data in tqdm(tail_data_loader):
    rank_rs = runEval.validation_step(data, 2)
    tail_list_result.append(rank_rs)
    # break

kq = runEval.validation_epoch_end((head_list_result, tail_list_result))


# EditedT5
<!--  -->

In [ ]:
from transformers import T5ForConditionalGeneration
from transformers import T5Config
import torch
import torch.nn as nn
from typing import Optional, Tuple
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import Seq2SeqLMOutput, BaseModelOutput


def shape(states, batch_size):
    return states.view(batch_size, -1, 8, 64).transpose(1, 2)


class EditedT5(T5ForConditionalGeneration):
  def __init__(self, config: T5Config):
    super().__init__(config)
    self.cross_dim = 256
    self.key_projection = nn.Linear(self.cross_dim, config.d_model)
    self.value_projection1 = nn.Linear(256+128, 256*4)
    self.value_projection2 = nn.Linear(256*4, config.d_model)

    self.post_init()

  def k_projection(self, x, i, batch_size):
    return shape(self.decoder.block[i].layer[1].EncDecAttention.k(x), batch_size)

  def v_projection(self, x, i, batch_size):
    return shape(self.decoder.block[i].layer[1].EncDecAttention.v(x), batch_size)

  def forward(
      self,
      input_ids: Optional[torch.LongTensor] = None,
      attention_mask: Optional[torch.FloatTensor] = None,
      decoder_input_ids: Optional[torch.LongTensor] = None,
      decoder_attention_mask: Optional[torch.BoolTensor] = None,
      head_mask: Optional[torch.FloatTensor] = None,
      decoder_head_mask: Optional[torch.FloatTensor] = None,
      cross_attn_head_mask: Optional[torch.Tensor] = None,
      encoder_outputs: Optional[Tuple[Tuple[torch.Tensor]]] = None,
      past_key_values: Optional[Tuple[Tuple[torch.Tensor]]] = None,
      inputs_embeds: Optional[torch.FloatTensor] = None,
      decoder_inputs_embeds: Optional[torch.FloatTensor] = None,
      labels: Optional[torch.LongTensor] = None,
      use_cache: Optional[bool] = None,
      output_attentions: Optional[bool] = None,
      output_hidden_states: Optional[bool] = None,
      return_dict: Optional[bool] = None,
      target_ent_embeddings=None,
      neighboors_embeddings=None,
      neighboors_embeddings_mask=None,
  ) :

      value_embeddings = self.value_projection2(F.prelu(self.value_projection1(neighboors_embeddings)))
      key_embeddings = self.key_projection(target_ent_embeddings) * neighboors_embeddings_mask.unsqueeze(2)
      batch_size = value_embeddings.shape[0]

      use_cache = use_cache if use_cache is not None else self.config.use_cache
      return_dict = return_dict if return_dict is not None else self.config.use_return_dict

      if encoder_outputs is None:
        # Convert encoder inputs in embeddings if needed
        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            inputs_embeds=inputs_embeds,
            head_mask=head_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
      elif return_dict and not isinstance(encoder_outputs, BaseModelOutput):
          encoder_outputs = BaseModelOutput(
              last_hidden_state=encoder_outputs[0],
              hidden_states=encoder_outputs[1] if len(encoder_outputs) > 1 else None,
              attentions=encoder_outputs[2] if len(encoder_outputs) > 2 else None,
          )
      hidden_states = encoder_outputs[0]

      if past_key_values is None:

        past_key_values = []
        for i in range(self.config.num_layers):
          cross_key = self.k_projection(key_embeddings, i, batch_size)
          cross_value = self.v_projection(value_embeddings, i, batch_size)
          self_key = torch.zeros_like(cross_key)
          self_value = torch.zeros_like(cross_value)
          past_key_values.append((self_key, self_value, cross_key, cross_value))

      if self.model_parallel:
          torch.cuda.set_device(self.decoder.first_device)

      if labels is not None and decoder_input_ids is None and decoder_inputs_embeds is None:
          decoder_input_ids = self._shift_right(labels)

      if self.model_parallel:
          torch.cuda.set_device(self.decoder.first_device)
          hidden_states = hidden_states.to(self.decoder.first_device)
          if decoder_input_ids is not None:
              decoder_input_ids = decoder_input_ids.to(self.decoder.first_device)
          if attention_mask is not None:
              attention_mask = attention_mask.to(self.decoder.first_device)
          if decoder_attention_mask is not None:
              decoder_attention_mask = decoder_attention_mask.to(self.decoder.first_device)

      # Decode
      decoder_outputs = self.decoder(
          input_ids=decoder_input_ids,
          attention_mask=decoder_attention_mask,
          inputs_embeds=decoder_inputs_embeds,
          past_key_values=past_key_values,
          encoder_hidden_states=hidden_states,
          encoder_attention_mask=attention_mask,
          head_mask=decoder_head_mask,
          cross_attn_head_mask=cross_attn_head_mask,
          use_cache=use_cache,
          output_attentions=output_attentions,
          output_hidden_states=output_hidden_states,
          return_dict=return_dict,
      )

      sequence_output = decoder_outputs[0]

      # Set device for model parallelism
      if self.model_parallel:
          torch.cuda.set_device(self.encoder.first_device)
          self.lm_head = self.lm_head.to(self.encoder.first_device)
          sequence_output = sequence_output.to(self.lm_head.weight.device)

      if self.config.tie_word_embeddings:
          sequence_output = sequence_output * (self.model_dim**-0.5)

      lm_logits = self.lm_head(sequence_output)

      loss = None
      if labels is not None:
          loss_fct = CrossEntropyLoss(ignore_index=-100)
          # move labels to correct device to enable PP
          labels = labels.to(lm_logits.device)
          loss = loss_fct(lm_logits.view(-1, lm_logits.size(-1)), labels.view(-1))

      if not return_dict:
          output = (lm_logits,) + decoder_outputs[1:] + encoder_outputs
          return ((loss,) + output) if loss is not None else output

      return Seq2SeqLMOutput(
          loss=loss,
          logits=lm_logits,
          past_key_values=decoder_outputs.past_key_values,
          decoder_hidden_states=decoder_outputs.hidden_states,
          decoder_attentions=decoder_outputs.attentions,
          cross_attentions=decoder_outputs.cross_attentions,
          encoder_last_hidden_state=encoder_outputs.last_hidden_state,
          encoder_hidden_states=encoder_outputs.hidden_states,
          encoder_attentions=encoder_outputs.attentions,
      )

  def prepare_inputs_for_generation(
        self,
        input_ids,
        past_key_values=None,
        attention_mask=None,
        head_mask=None,
        decoder_head_mask=None,
        decoder_attention_mask=None,
        cross_attn_head_mask=None,
        use_cache=None,
        encoder_outputs=None,
        target_ent_embeddings=None,
        neighboors_embeddings=None,
        neighboors_embeddings_mask=None,
        **kwargs,
    ):
        # cut decoder_input_ids if past_key_values is used
        if past_key_values is not None:
            past_length = past_key_values[0][0].shape[2]

            # Some generation methods already pass only the last input ID
            if input_ids.shape[1] > past_length:
                remove_prefix_length = past_length
            else:
                # Default to old behavior: keep only final ID
                remove_prefix_length = input_ids.shape[1] - 1

            input_ids = input_ids[:, remove_prefix_length:]

        return {
            "decoder_input_ids": input_ids,
            "past_key_values": past_key_values,
            "encoder_outputs": encoder_outputs,
            "attention_mask": attention_mask,
            "head_mask": head_mask,
            "decoder_head_mask": decoder_head_mask,
            "decoder_attention_mask": decoder_attention_mask,
            "cross_attn_head_mask": cross_attn_head_mask,
            "use_cache": use_cache,
            "target_ent_embeddings": target_ent_embeddings,
            "neighboors_embeddings": neighboors_embeddings,
            "neighboors_embeddings_mask": neighboors_embeddings_mask,
            # "input_ids": input_ids,
        }

# write dataset


In [ ]:
!pip install ampligraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.3/210.3 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.5/575.5 kB 25.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 60.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 33.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 40.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.5/84.5 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.4/561.4 kB 36.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.9 MB/s 

In [ ]:
# from ampligraph.datasets import load_fb15k_237
# from ampligraph.pretrained_models import load_pretrained_model

# model = load_pretrained_model(dataset="wn18rr", scoring_type="RotatE")

# # dataset/wn18rr

In [ ]:
# # ent2id
# from tqdm.auto import tqdm
# import os

# if not os.path.exists('/content/entity2id.txt'):
#   !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/entity2id.txt
# path = "/content/entity2id.txt"

# ent2id = {}
# with open(path, "r") as f:
#   total_lines = sum(1 for _ in f)
#   f.seek(0)  # Reset file pointer to the beginning
#   for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
#     if i==0:
#       continue
#     # print(line.strip().split('\t'))
#     ent, id = line.strip().split('\t')
#     ent2id[ent] = int(id)


In [ ]:
# # rel2id
# from tqdm.auto import tqdm
# import os

# if not os.path.exists('/content/relation2id.txt'):
#   !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/relation2id.txt
# path = "/content/relation2id.txt"

# rel2id = {}
# with open(path, "r") as f:
#   total_lines = sum(1 for _ in f)
#   f.seek(0)  # Reset file pointer to the beginning
#   for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
#     if i==0:
#       continue
#     rel, id = line.strip().split('\t')
#     rel2id[rel] = int(id)


In [ ]:
%%capture
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained('t5-small', padding=True)

def _tokenize( x):
    return tokenizer(x, return_tensors="pt")['input_ids'][0][:-1]

In [ ]:
# ent2text
from tqdm.auto import tqdm
import os

if not os.path.exists('/content/entityid2name.txt'):
  !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/entityid2name.txt
path = "/content/entityid2name.txt"

ent2text = {}
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    ent, text = line.strip().split('\t')
    ename, etype, _ = text.split(',')
    text = etype + ', ' + ename + ', '
    ent2text[ent] = _tokenize(text)


--2024-04-21 15:23:06--  https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/entityid2name.txt
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/entityid2name.txt [following]
--2024-04-21 15:23:06--  https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/entityid2name.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1044056 (1020K) [text/plain]
Saving to: ‘entityid2name.txt’

entityid2name.txt   100%[===================>]   1020K  --.-KB/s    in 0.05s   

2024-04-21 15:23:06 (20.6 MB/s) - ‘entity

Processing lines:   0%|          | 0/40944 [00:00<?, ?it/s]

In [ ]:
# rel2text
from tqdm.auto import tqdm
import os

if not os.path.exists('/content/relationid2name.txt'):
  !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/relationid2name.txt
path = "/content/relationid2name.txt"

rel2text = {}
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    rel, text = line.strip().split('\t')
    rel2text[rel] = _tokenize(text)


--2024-04-21 15:23:14--  https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/relationid2name.txt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/relationid2name.txt [following]
--2024-04-21 15:23:14--  https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/relationid2name.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 206 [text/plain]
Saving to: ‘relationid2name.txt’

relationid2name.txt 100%[===================>]     206  --.-KB/s    in 0s      

2024-04-21 15:23:14 (5.15 MB/s) - ‘relationid

Processing lines:   0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
# ent2decs
from tqdm.auto import tqdm
import os

if not os.path.exists('/content/entityid2description.txt'):
  !wget https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/entityid2description.txt
path = "/content/entityid2description.txt"

ent2decs = {}
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    ent, text = line.strip().split('\t')
    ent2decs[ent] = _tokenize(text)[:64]


--2024-04-21 15:23:14--  https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/entityid2description.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3553011 (3.4M) [text/plain]
Saving to: ‘entityid2description.txt’

entityid2descriptio 100%[===================>]   3.39M  --.-KB/s    in 0.07s   

2024-04-21 15:23:15 (48.0 MB/s) - ‘entityid2description.txt’ saved [3553011/3553011]



Processing lines:   0%|          | 0/40944 [00:00<?, ?it/s]

In [ ]:
# train triplet
from tqdm.auto import tqdm
import torch
import os

if not os.path.exists('/content/train2id.txt'):
  !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/train2id.txt
path = "/content/train2id.txt"

train_triplet_id = []
train_triplet_tokens = []
# train_triplet_decs = []
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    head_id, tail_id, relation_id = line.strip().split(' ')
    head_tokens = ent2text[head_id]
    relation_tokens = rel2text[relation_id]
    tail_tokens = ent2text[tail_id]
    head_decs = ent2decs[head_id]
    tail_decs = ent2decs[tail_id]
    head_tokens = torch.cat([head_tokens, head_decs])
    tail_tokens = torch.cat([tail_tokens, tail_decs])
    head_id = int(head_id)
    relation_id = int(relation_id)
    tail_id = int(tail_id)
    train_triplet_id.append((head_id, relation_id, tail_id))
    train_triplet_tokens.append((head_tokens, relation_tokens, tail_tokens))
    # train_triplet_decs.append((head_decs, tail_decs))

Processing lines:   0%|          | 0/86836 [00:00<?, ?it/s]

In [ ]:
tokenizer.decode(train_triplet_tokens[0][0])

'NN, land reform, a redistribution of agricultural land (especially by government action)'

In [ ]:
# id2ent = {v: k for k, v in ent2id.items()}
# id2rel = {v: k for k, v in rel2id.items()}

In [ ]:
# valid triplet
from tqdm.auto import tqdm
import os

if not os.path.exists('/content/valid2id.txt'):
  !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/valid2id.txt
path = "/content/valid2id.txt"

valid_triplet_id = []
valid_triplet_tokens = []
# valid_triplet_decs = []
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    head_id, tail_id, relation_id = line.strip().split(' ')
    head_tokens = ent2text[head_id]
    relation_tokens = rel2text[relation_id]
    tail_tokens = ent2text[tail_id]
    head_decs = ent2decs[head_id]
    tail_decs = ent2decs[tail_id]
    head_tokens = torch.cat([head_tokens, head_decs])
    tail_tokens = torch.cat([tail_tokens, tail_decs])
    head_id = int(head_id)
    relation_id = int(relation_id)
    tail_id = int(tail_id)
    valid_triplet_id.append((head_id, relation_id, tail_id))
    valid_triplet_tokens.append((head_tokens, relation_tokens, tail_tokens))
    # valid_triplet_decs.append((head_decs, tail_decs))

--2024-04-21 15:23:48--  https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/valid2id.txt
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/valid2id.txt [following]
--2024-04-21 15:23:48--  https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/valid2id.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40861 (40K) [text/plain]
Saving to: ‘valid2id.txt’

valid2id.txt        100%[===================>]  39.90K  --.-KB/s    in 0.003s  

2024-04-21 15:23:48 (11.6 MB/s) - ‘valid2id.txt’ saved [40861/408

Processing lines:   0%|          | 0/3035 [00:00<?, ?it/s]

In [ ]:
# test triplet
from tqdm.auto import tqdm
import os

if not os.path.exists('/content/test2id.txt'):
  !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/test2id.txt
path = "/content/test2id.txt"

test_triplet_id = []
test_triplet_tokens = []
# test_triplet_decs = []
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    head_id, tail_id, relation_id = line.strip().split(' ')
    head_tokens = ent2text[head_id]
    relation_tokens = rel2text[relation_id]
    tail_tokens = ent2text[tail_id]
    head_decs = ent2decs[head_id]
    tail_decs = ent2decs[tail_id]
    head_tokens = torch.cat([head_tokens, head_decs])
    tail_tokens = torch.cat([tail_tokens, tail_decs])
    head_id = int(head_id)
    relation_id = int(relation_id)
    tail_id = int(tail_id)
    test_triplet_id.append((head_id, relation_id, tail_id))
    test_triplet_tokens.append((head_tokens, relation_tokens, tail_tokens))
    # test_triplet_decs.append((head_decs, tail_decs))

--2024-04-21 15:23:48--  https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/test2id.txt
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/test2id.txt [following]
--2024-04-21 15:23:49--  https://raw.githubusercontent.com/chenchens190009/KG-S2S/main/data/processed/WN18RR/test2id.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 42316 (41K) [text/plain]
Saving to: ‘test2id.txt’

test2id.txt         100%[===================>]  41.32K  --.-KB/s    in 0.01s   

2024-04-21 15:23:49 (3.85 MB/s) - ‘test2id.txt’ saved [42316/42316]



Processing lines:   0%|          | 0/3135 [00:00<?, ?it/s]

In [ ]:

# rel2id
from tqdm.auto import tqdm
import os


path = "/content/relation_ids.del"

rel2id = {}
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    id, rel = line.strip().split('\t')
    rel2id[rel] = int(id)


Processing lines:   0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
# ent2id
from tqdm.auto import tqdm
import os

path = "/content/entity_ids.del"

ent2id = {}
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    # print(line.strip().split('\t'))
    id, ent = line.strip().split('\t')
    ent2id[ent] = int(id)


Processing lines:   0%|          | 0/40943 [00:00<?, ?it/s]

In [ ]:
# ent2id
from tqdm.auto import tqdm
import os

if not os.path.exists('/content/entity2id.txt'):
  !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/entity2id.txt
path = "/content/entity2id.txt"

old_ent2id = {}
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    # print(line.strip().split('\t'))
    ent, id = line.strip().split('\t')
    old_ent2id[ent] = int(id)


Processing lines:   0%|          | 0/40944 [00:00<?, ?it/s]

In [ ]:
# rel2id
from tqdm.auto import tqdm
import os

if not os.path.exists('/content/relation2id.txt'):
  !wget https://github.com/chenchens190009/KG-S2S/raw/main/data/processed/WN18RR/relation2id.txt
path = "/content/relation2id.txt"

old_rel2id = {}
with open(path, "r") as f:
  total_lines = sum(1 for _ in f)
  f.seek(0)  # Reset file pointer to the beginning
  for i, line in tqdm(enumerate(f), total=total_lines, desc="Processing lines"):
    if i==0:
      continue
    rel, id = line.strip().split('\t')
    old_rel2id[rel] = int(id)


Processing lines:   0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
rel2id

{'_hypernym': 0,
 '_derivationally_related_form': 1,
 '_instance_hypernym': 2,
 '_also_see': 3,
 '_member_meronym': 4,
 '_synset_domain_topic_of': 5,
 '_has_part': 6,
 '_member_of_domain_usage': 7,
 '_member_of_domain_region': 8,
 '_verb_group': 9,
 '_similar_to': 10}

In [ ]:
import torch

TransE_ent = torch.tensor(new_rotate_model['_entity_embedder'].numpy())

new_TransE_ent_dict = {}
for rent, rid in tqdm(ent2id.items()):
  emb = TransE_ent[rid]
  ent_id = old_ent2id[rent]
  new_TransE_ent_dict[ent_id] = emb

new_TransE_ent = []
for i in range(len(ent2id)):
  if i in new_TransE_ent_dict:
    new_TransE_ent.append(new_TransE_ent_dict[i])

new_TransE_ent = torch.stack(new_TransE_ent)

TransE_rel = torch.tensor(new_rotate_model['_relation_embedder'].numpy())

new_TransE_rel_dict = {}
for rrel, rid  in rel2id.items():
  emb = TransE_rel[rid]
  rel_id = old_rel2id[rrel]
  new_TransE_rel_dict[rel_id] = emb

new_TransE_rel = []
for i in range(len(rel2id)):
  if i in new_TransE_rel_dict:
    new_TransE_rel.append(new_TransE_rel_dict[i])

new_TransE_rel = torch.stack(new_TransE_rel)

  0%|          | 0/40943 [00:00<?, ?it/s]

In [ ]:
new_TransE_rel.shape

torch.Size([11, 128])

In [ ]:
len(rel2id)

11

In [ ]:
!gsutil cp   gs://hien7613storage2/wnrr-rotate.pt /content/

Copying gs://hien7613storage2/wnrr-rotate.pt...
\ [1 files][ 40.0 MiB/ 40.0 MiB]                                                
Operation completed over 1 objects/40.0 MiB.                                     


In [ ]:
import torch
new_rotate_model = torch.load("/content/wnrr-rotate.pt")

In [ ]:
new_rotate_model['_entity_embedder']

tensor([[-0.8534,  0.5461,  0.4359,  ..., -0.1295,  0.5124, -0.3342],
        [ 0.5979, -0.1284,  0.3372,  ..., -0.7183,  0.3614,  0.1950],
        [ 0.0849, -0.6299, -0.3926,  ..., -0.5062, -0.3987,  0.3825],
        ...,
        [ 0.0208,  0.0495, -0.0272,  ..., -0.0808,  0.1049, -0.2479],
        [ 0.0272, -0.0217, -0.0520,  ..., -0.0851,  0.0329, -0.2540],
        [ 0.0209,  0.0378, -0.0488,  ..., -0.0554,  0.1309, -0.2811]])

In [ ]:
new_rotate_model['_relation_embedder'].shape

torch.Size([22, 128])

In [ ]:
kgt5_wn18rr_data = {
    "train_triplet_id":torch.tensor(train_triplet_id),
    "train_triplet_tokens":train_triplet_tokens,
    "valid_triplet_id":torch.tensor(valid_triplet_id),
    "valid_triplet_tokens":valid_triplet_tokens,
    "test_triplet_id":torch.tensor(test_triplet_id),
    "test_triplet_tokens":test_triplet_tokens,
    "struct_ent_emb":new_TransE_ent,
    "struct_rel_emb":new_TransE_rel,
}

In [ ]:
import torch
torch.save(kgt5_wn18rr_data, "/content/kgt5_wn18rr_data.pt")
!gsutil cp kgt5_wn18rr_data.pt gs://hien7613storage2/

In [ ]:
kgt5_wn18rr_data.keys()